In [1]:
import shutil
shutil.rmtree("cache", ignore_errors=True)

# Mapping the Faithfulness of Chain-of-Thought Reasoning

**Research Question**: Can we build a comprehensive taxonomy of CoT failure modes and develop information-theoretic metrics that predict when reasoning steps add genuine signal versus noise?

---

## Pipeline Overview
1. **Setup & Configuration** — Install dependencies, configure paths
2. **Data Download & Parsing** — Download 5 benchmarks, parse into unified format
3. **Preprocessing** — Create splits, format prompts
4. **Baseline Accuracy** — Run all 5 models on all 5 benchmarks
5. **Faithfulness Profiling** — Compute SIG, CNS, RFI metrics
6. **Perturbation Tests** — Early answering, mistake injection, shuffling, deletion, paraphrasing
7. **Failure Taxonomy** — Classify CoT failure modes
8. **Ablation Studies** — Temperature, length, perturbation type, prompt format, model scaling
9. **Cross-Model Analysis** — Inverse scaling hypothesis
10. **Results & Tables** — Generate formatted tables
11. **Visualization** — Heatmaps, radar charts, scaling curves, step-level plots
12. **Sampling** — Sample and inspect individual CoT examples

## 1. Setup & Configuration

In [2]:
# Install dependencies (run once)
!pip install -r requirements.txt
#!pip install torch torchaudio torchvision numpy datasets pandas matplotlib tqdm typing seaborn dataclasses dotenv transformers

In [3]:
!pip install packaging ninja

In [4]:
!pip install flash-attn --no-build-isolation --no-cache-dir

In [5]:
import flash_attn
print("flash-attn works")

flash-attn works


In [6]:
import os
import sys
import json
import random
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is in path
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import configs
from configs.model_config import MODEL_REGISTRY, GENERATION_CONFIG, get_model_config
from configs.benchmark_config import BENCHMARK_REGISTRY, get_benchmark_config
from configs.experiment_config import EXPERIMENT_CONFIG, PATHS, ensure_dirs

# Create all directories
ensure_dirs()

# Set seeds for reproducibility
SEED = EXPERIMENT_CONFIG['seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print(f'Project Root: {PROJECT_ROOT}')
print(f'Device: {EXPERIMENT_CONFIG["device"]}')
print(f'Precision: {EXPERIMENT_CONFIG["dtype"]}')
print(f'AMP: {EXPERIMENT_CONFIG["use_amp"]}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'\nModels: {list(MODEL_REGISTRY.keys())}')
print(f'Benchmarks: {list(BENCHMARK_REGISTRY.keys())}')

Project Root: /teamspace/studios/this_studio
Device: cuda
Precision: bfloat16
AMP: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB

Models: ['ds-r1-qwen-7b', 'ds-r1-llama-8b', 'ds-r1-qwen-14b', 'qwq-32b', 'ds-r1-qwen-32b']
Benchmarks: ['gsm8k', 'math', 'strategyqa', 'arc_challenge', 'folio']


In [7]:
import sys
sys.path.insert(0, "./")

In [8]:
os.getcwd()

'/teamspace/studios/this_studio'

In [ ]:
from huggingface_hub import login

login("")

## 2. Data Download & Parsing

In [10]:
from src.data.download_datasets import download_all_datasets

# Download all 5 benchmark datasets
download_info = download_all_datasets(
    output_dir=PATHS['raw_data_dir'],
    cache_dir=PATHS['cache_dir'],
)

print("\n=== Download Summary ===")

for bench, info in download_info.items():
    print(f"\n{bench}:")

    if "error" in info:
        print(f"  ❌ {info['error']}")
        continue

    splits = info.get("splits", [])
    sizes = info.get("sizes", {})

    # SAFE handling
    if isinstance(splits, list):
        for s in splits:
            print(f"  {bench}/{s}: {sizes.get(s, '?')} examples")
    else:
        print(f"  {bench}: invalid splits format -> {type(splits)}")

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

  ✅ Done: ['train', 'test']
    train: 7473
    test: 1319



Generating train split:   0%|          | 0/12500 [00:00<?, ? examples/s]

  ✅ Done: ['train']
    train: 12500



...

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

  ✅ Done: ['train', 'test', 'validation']
    train: 1119
    test: 1172
    validation: 299



Generating train split:   0%|          | 0/1001 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/203 [00:00<?, ? examples/s]

  ✅ Done: ['train', 'validation']
    train: 1001
    validation: 203

=== Summary ===
gsm8k: ['train', 'test']
math: ['train']
strategyqa: ['train', 'test']
arc-challenge: ['train', 'test', 'validation']
folio: ['train', 'validation']

=== Download Summary ===

gsm8k:
  gsm8k/train: 7473 examples
  gsm8k/test: 1319 examples

math:
  math/train: 12500 examples

strategyqa:
  strategyqa/train: 1603 examples
  strategyqa/test: 687 examples

arc-challenge:
  arc-challenge/train: 1119 examples
  arc-challenge/test: 1172 examples
  arc-challenge/validation: 299 examples

folio:
  folio/train: 1001 examples
  folio/validation: 203 examples


In [11]:
from datasets import load_dataset
import re
from src.data.dataset_loader import DatasetLoader
from src.data.preprocessing import preprocess_dataset
from configs.benchmark_config import BENCHMARK_REGISTRY

processed_data = {}

for bench_key, config in BENCHMARK_REGISTRY.items():
    print(f"\nLoading {bench_key}...")

    loader = DatasetLoader(config)
    data = loader.load()

    data = data.select(range(min(200, len(data))))  # subsample

    processed = preprocess_dataset(data, bench_key, "zero_shot_cot")

    processed_data[bench_key] = processed

print("\nprocessed_data ready:")
for k, v in processed_data.items():
    print(f"{k}: {len(v)} examples")


Loading gsm8k...


Loaded openai/gsm8k [test] → 1319 samples

Loading math...
Loaded qwedsacf/competition_math [train] → 12500 samples

Loading strategyqa...
Loaded ChilleD/StrategyQA [test] → 687 samples

Loading arc_challenge...
Loaded allenai/ai2_arc [test] → 1172 samples

Loading folio...
Loaded tasksource/folio [validation] → 203 samples

processed_data ready:
gsm8k: 200 examples
math: 200 examples
strategyqa: 200 examples
arc_challenge: 200 examples
folio: 200 examples


In [12]:
from concurrent.futures import ThreadPoolExecutor
from src.models.model_loader import ModelManager
from src.models.inference import InferenceEngine
from src.utils.answer_extractor import AnswerExtractor
from src.utils.cache import get_cache, save_cache, _hash
import time
import os, json

def run_model(model_key):
    model_config = MODEL_REGISTRY[model_key]

    print(f'\n{"="*60}')
    print(f'Model: {model_config.short_name} ({model_config.params_b}B)')
    print(f'{"="*60}')

    model_manager = ModelManager(cache_dir=PATHS.get('model_cache_dir'))
    answer_extractor = AnswerExtractor()

    model, tokenizer = model_manager.load_model(model_config)
    gen_config = model_manager.get_generation_config()

    engine = InferenceEngine(
        model, tokenizer, gen_config,
        device=EXPERIMENT_CONFIG['device'],
        use_amp=EXPERIMENT_CONFIG['use_amp'],
    )

    model_results = {}

    for bench_key, examples in processed_data.items():
        correct = 0
        predictions = []
        
        prompts = [ex['prompt'] for ex in examples]

        batch_size = min(model_config.batch_size, len(prompts))
        print(f"[{bench_key}] Batch size: {batch_size}")

        # Cache
        cache_hits = {}
        new_prompts = []
        new_indices = []
        cache_hit_count = 0

        for i, prompt in enumerate(prompts):
            key = _hash(prompt)
            cached = get_cache(key)

            if cached:
                cache_hits[i] = cached
                cache_hit_count += 1
            else:
                new_prompts.append(prompt)
                new_indices.append(i)

        print(f"[{bench_key}] Cache hits: {cache_hit_count}/{len(prompts)}")

        if new_prompts:
            print(f"[{bench_key}] Generating {len(new_prompts)} new samples...")
            start = time.time()

            new_outputs = engine.generate_batch(
                new_prompts,
                batch_size=batch_size
            )

            print(f"[{bench_key}] Generation time: {time.time() - start:.2f}s")

            for idx, out in zip(new_indices, new_outputs):
                key = _hash(prompts[idx])

                safe_out = {
                    "raw_output": out["raw_output"],
                    "num_steps": out["num_steps"],
                    "num_generated_tokens": out["num_generated_tokens"],
                }

                save_cache(key, safe_out)
                cache_hits[idx] = safe_out

        outputs = [cache_hits[i] for i in range(len(prompts))]

        print(f"[{bench_key}] Outputs ready: {len(outputs)}")

        for i, (ex, output) in enumerate(zip(examples, outputs)):

            if i < 2:
                print("\n================ DEBUG SAMPLE ================")
                print(f"Benchmark: {bench_key}")
                print(f"Index: {i}")
                print("\n--- INPUT ---")
                print("QUESTION:", ex.get("question", "")[:200])
                print("\n--- MODEL OUTPUT ---")
                print(output.get("raw_output", "")[:300])

            #  FIXED ORDER
            predicted = engine.extract_answer(
                output['raw_output'], ex['answer_type'], bench_key
            )

            is_correct = answer_extractor.check_answer(
                predicted, ex['gold_answer'], ex['answer_type']
            )

            if i < 2:
                print("\n--- EXTRACTED ---")
                print("PRED:", predicted)
                print("GOLD:", ex['gold_answer'])
                print("CORRECT:", is_correct)
                print("=============================================")

            if is_correct:
                correct += 1

            if (i + 1) % 50 == 0:
                print(f'  {bench_key}: {i+1}/{len(examples)}, Acc: {correct/(i+1)*100:.1f}%')

            #  LIVE ACCURACY
            #if (i + 1) % 20 == 0:
                #print(f"[{bench_key}] {i+1}/{len(examples)} → Acc: {correct/(i+1)*100:.2f}%")

            predictions.append({
                'id': ex.get('id', f'{bench_key}_{i}'),
                'gold_answer': ex['gold_answer'],
                'predicted_answer': predicted,
                'is_correct': is_correct,
                'raw_output': output['raw_output'][:500],
                'num_steps': output['num_steps'],
                'num_tokens': output['num_generated_tokens'],
            })

        accuracy = correct / max(1, len(examples)) * 100
        model_results[bench_key] = {
            'accuracy': accuracy,
            'correct': correct,
            'total': len(examples),
            'predictions': predictions,
        }

        print(f'  {bench_key}: {accuracy:.2f}%')

    # Save
    os.makedirs(os.path.join(PATHS['raw_results_dir'], 'baseline'), exist_ok=True)

    with open(os.path.join(PATHS['raw_results_dir'], 'baseline', f'baseline_{model_key}.json'), 'w') as f:
        json.dump({'model': model_config.short_name, 'results': model_results}, f, indent=2)

    with open(os.path.join(PATHS['raw_results_dir'], 'baseline', f'predictions_{model_key}.json'), 'w') as f:
        json.dump({'model': model_config.short_name, 'predictions': {
            k: v['predictions'] for k, v in model_results.items()
        }}, f, indent=2)

    model_manager.unload_model()
    return model_results

In [13]:
# ================= RUN =================
baseline_results = {}

MAX_WORKERS = 1

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results = list(executor.map(run_model, MODEL_REGISTRY.keys()))

for model_key, res in zip(MODEL_REGISTRY.keys(), results):
    baseline_results[model_key] = res

print('\n=== Baseline Complete ===')


Model: DS-R1-Qwen-7B (7.0B)
[2026-04-16 00:54:09] [INFO] [model_loader] Loading model DS-R1-Qwen-7B (deepseek-ai/DeepSeek-R1-Distill-Qwen-7B) in bfloat16...


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[2026-04-16 01:55:17] [INFO] [model_loader] GPU Memory: 61.0GB allocated, 61.1GB reserved
[2026-04-16 01:55:17] [INFO] [model_loader] Successfully loaded DS-R1-Qwen-32B
[gsm8k] Batch size: 8
[gsm8k] Cache hits: 200/200
[gsm8k] Outputs ready: 200

================ DEBUG SAMPLE ================
Benchmark: gsm8k
Index: 0

--- INPUT ---
QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fre

--- MODEL OUTPUT ---
Determine the total number of eggs Janet's ducks lay per day. Step 2: Calculate how many eggs Janet consumes or uses for baking each day. Step 3: Subtract the consumed eggs from the total to find the remaining eggs. Step 4: Calculate the revenue Janet makes from selling the remaining eggs. Step 5: S

--- EXTRACTED ---
PRED: 16
GOLD: 18
CORRECT: False

================ DEBUG SAMPLE ================
Benchmark: gsm8k
Index: 1

--- 

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[2026-04-16 01:55:24] [INFO] [model_loader] GPU Memory: 14.2GB allocated, 61.1GB reserved
[2026-04-16 01:55:24] [INFO] [model_loader] Successfully loaded DS-R1-Qwen-7B
[2026-04-16 01:55:24] [INFO] [exp_faithfulness]   Processing gsm8k (200 examples)
[2026-04-16 01:55:24] [INFO] [exp_faithfulness]     Computing SIG...
[2026-04-16 01:55:24] [WARNING] [sig_metric] SIG computation failed for example 0: cannot reshape tensor of 0 elements into shape [1, 0, -1, 128] because the unspecified dimension size -1 can be any value and is ambiguous
[2026-04-16 01:55:24] [WARNING] [sig_metric] SIG computation failed for example 1: cannot reshape tensor of 0 elements into shape [1, 0, -1, 128] because the unspecified dimension size -1 can be any value and is ambiguous
[2026-04-16 01:55:24] [WARNING] [sig_metric] SIG computation failed for example 2: cannot reshape tensor of 0 elements into shape [1, 0, -1, 128] because the unspecified dimension size -1 can be any value and is ambiguous
[2026-04-16 01:

KeyboardInterrupt: 

## 5. Faithfulness Profiling (Experiment 2)

Compute SIG, CNS, and RFI metrics for all models.

In [ ]:
# # ================= FAITHFULNESS =================
# from scripts.experiments.exp_faithfulness_profiling import run_faithfulness_profiling

# faithfulness_results = run_faithfulness_profiling()

# print("\n=== FULL RESULTS ===")

# for model_key in MODEL_REGISTRY:
#     model_name = MODEL_REGISTRY[model_key].short_name
#     print(f"\n{model_name}")

#     for bench_key in BENCHMARK_REGISTRY:
#         acc = baseline_results[model_key][bench_key]["accuracy"]

#         rfi = faithfulness_results[model_key][bench_key]\
#             .get("rfi_aggregate", {})\
#             .get("mean_rfi", 0)

#         print(f"{bench_key}: Acc={acc:.2f} | RFI={rfi:.4f}")

In [14]:
from scripts.experiments.exp_faithfulness_profiling import run_faithfulness_profiling

faithfulness_results = run_faithfulness_profiling()

print('\n=== Faithfulness Profiling Complete ===')
for model_key, benchmarks in faithfulness_results.items():
    print(f'\n{MODEL_REGISTRY[model_key].short_name}:')
    for bench_key, result in benchmarks.items():
        rfi = result.get('rfi_aggregate', {}).get('mean_rfi', 0)
        print(f'  {bench_key}: RFI = {rfi:.4f}')

[2026-04-16 02:00:01] [INFO] [exp_faithfulness] === Faithfulness profiling: DS-R1-Qwen-7B ===
[2026-04-16 02:00:01] [INFO] [model_loader] Loading model DS-R1-Qwen-7B (deepseek-ai/DeepSeek-R1-Distill-Qwen-7B) in bfloat16...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[2026-04-16 02:00:03] [INFO] [model_loader] GPU Memory: 28.4GB allocated, 61.3GB reserved
[2026-04-16 02:00:03] [INFO] [model_loader] Successfully loaded DS-R1-Qwen-7B
[2026-04-16 02:00:03] [INFO] [exp_faithfulness]   Processing gsm8k (200 examples)
[2026-04-16 02:00:03] [INFO] [exp_faithfulness]     Computing SIG...
[2026-04-16 02:00:03] [WARNING] [sig_metric] SIG computation failed for example 0: cannot reshape tensor of 0 elements into shape [1, 0, -1, 128] because the unspecified dimension size -1 can be any value and is ambiguous
[2026-04-16 02:00:03] [WARNING] [sig_metric] SIG computation failed for example 1: cannot reshape tensor of 0 elements into shape [1, 0, -1, 128] because the unspecified dimension size -1 can be any value and is ambiguous
[2026-04-16 02:00:03] [WARNING] [sig_metric] SIG computation failed for example 2: cannot reshape tensor of 0 elements into shape [1, 0, -1, 128] because the unspecified dimension size -1 can be any value and is ambiguous
[2026-04-16 02:

KeyboardInterrupt: 

## 6. Perturbation Tests (Experiment 3)

In [ ]:
from scripts.experiments.exp_perturbation_tests import run_perturbation_tests

perturbation_results = run_perturbation_tests()
print('\n=== Perturbation Tests Complete ===')

## 7. Failure Taxonomy (Experiment 4)

In [ ]:
from scripts.experiments.exp_failure_classification import run_failure_classification

failure_results = run_failure_classification()
print('\n=== Failure Classification Complete ===')

## 8. Ablation Studies

Run all 5 ablation experiments.

In [ ]:
# Ablation 1: Temperature
from scripts.ablations.ablation_temperature import run_temperature_ablation
temp_results = run_temperature_ablation()
print('Temperature ablation complete.')

In [ ]:
# Ablation 2: CoT Length
from scripts.ablations.ablation_cot_length import run_cot_length_ablation
length_results = run_cot_length_ablation()
print('CoT length ablation complete.')

In [ ]:
# Ablation 3: Perturbation Type Comparison
from scripts.ablations.ablation_perturbation_type import run_perturbation_type_ablation
pert_type_results = run_perturbation_type_ablation()
print('Perturbation type ablation complete.')

In [ ]:
# Ablation 4: Prompt Format
from scripts.ablations.ablation_prompt_format import run_prompt_format_ablation
format_results = run_prompt_format_ablation()
print('Prompt format ablation complete.')

In [ ]:
# Ablation 5: Model Scaling
from scripts.ablations.ablation_model_scaling import run_model_scaling_analysis
scaling_results = run_model_scaling_analysis()
print('Model scaling ablation complete.')

## 9. Cross-Model Analysis (Experiment 5)

In [ ]:
from scripts.experiments.exp_cross_model_analysis import run_cross_model_analysis

cross_model_results = run_cross_model_analysis()

# Print key findings
inverse = cross_model_results.get('inverse_scaling_test', {})
print(f'\nInverse Scaling Test:')
print(f'  Correlation (params vs RFI): {inverse.get("correlation", "N/A")}')
print(f'  Detected: {inverse.get("inverse_scaling_detected", "N/A")}')
print(f'  Interpretation: {inverse.get("interpretation", "N/A")}')

## 10. Results & Tables

In [ ]:
from scripts.visualization.generate_tables import generate_all_tables

tables = generate_all_tables()
print(tables)

## 11. Visualization

In [ ]:
from scripts.visualization.plot_heatmaps import (
    plot_accuracy_heatmap, plot_faithfulness_heatmap, plot_perturbation_heatmap
)
from scripts.visualization.plot_radar_charts import plot_failure_radar, plot_step_type_radar
from scripts.visualization.plot_scaling_curves import plot_scaling_curves, plot_accuracy_faithfulness_scatter
from scripts.visualization.plot_step_information import plot_aggregate_step_info

# Generate all visualizations
plot_accuracy_heatmap()
plot_faithfulness_heatmap()
plot_perturbation_heatmap()
plot_failure_radar()
plot_step_type_radar()
plot_scaling_curves()
plot_accuracy_faithfulness_scatter()
plot_aggregate_step_info()

print(f'\nAll figures saved to: {PATHS["figures_dir"]}')

In [ ]:
# Display generated figures
from IPython.display import Image, display
import glob

fig_dir = PATHS['figures_dir']
for fig_path in sorted(glob.glob(os.path.join(fig_dir, '*.png'))):
    print(f'\n--- {os.path.basename(fig_path)} ---')
    display(Image(filename=fig_path, width=800))

## 12. Sampling & Inspection

Sample individual CoT examples to qualitatively inspect reasoning patterns.

In [ ]:
# Sample from baseline predictions
import random

def sample_and_display(model_key, bench_key, n=3, filter_correct=None):
    """Sample and display CoT examples."""
    pred_path = os.path.join(
        PATHS['raw_results_dir'], 'baseline', f'predictions_{model_key}.json'
    )
    if not os.path.exists(pred_path):
        print(f'No predictions found for {model_key}')
        return
    
    with open(pred_path) as f:
        pred_data = json.load(f)
    
    predictions = pred_data.get('predictions', {}).get(bench_key, [])
    
    if filter_correct is not None:
        predictions = [p for p in predictions if p['is_correct'] == filter_correct]
    
    if not predictions:
        print('No matching examples found.')
        return
    
    samples = random.sample(predictions, min(n, len(predictions)))
    
    model_name = MODEL_REGISTRY[model_key].short_name
    print(f'\n{"="*60}')
    print(f'Model: {model_name} | Benchmark: {bench_key}')
    print(f'Filter: {"correct" if filter_correct else "incorrect" if filter_correct is False else "all"}')
    print(f'{"="*60}')
    
    for i, sample in enumerate(samples):
        print(f'\n--- Sample {i+1} ---')
        print(f'Gold: {sample["gold_answer"]}')
        print(f'Predicted: {sample["predicted_answer"]}')
        print(f'Correct: {sample["is_correct"]}')
        print(f'Steps: {sample["num_steps"]}, Tokens: {sample["num_tokens"]}')
        print(f'\nReasoning:\n{sample["raw_output"]}')
        print()

# Sample correct examples
for model_key in list(MODEL_REGISTRY.keys())[:2]:  # First 2 models
    sample_and_display(model_key, 'gsm8k', n=2, filter_correct=True)
    sample_and_display(model_key, 'gsm8k', n=2, filter_correct=False)

In [ ]:
# Analyze CoT structure of sampled examples
from src.utils.cot_parser import CoTParser

parser = CoTParser()

def analyze_cot_structure(raw_output):
    """Analyze and display CoT structure."""
    parsed = parser.parse(raw_output)
    print(f'Total Steps: {parsed.num_steps}')
    print(f'Has Think Tags: {parsed.has_think_tags}')
    print(f'Final Answer: {parsed.final_answer_text}')
    print('\nStep Breakdown:')
    for step in parsed.steps:
        print(f'  [{step.step_type.upper():15s}] {step.text[:80]}...')
    return parsed

# Analyze a sample
pred_path = os.path.join(PATHS['raw_results_dir'], 'baseline', 'predictions_ds-r1-qwen-7b.json')
if os.path.exists(pred_path):
    with open(pred_path) as f:
        preds = json.load(f)
    gsm8k_preds = preds.get('predictions', {}).get('gsm8k', [])
    if gsm8k_preds:
        print('=== CoT Structure Analysis ===')
        analyze_cot_structure(gsm8k_preds[0]['raw_output'])

## Summary

This notebook runs the complete **FaithCoT** pipeline:
- **5 Benchmarks**: GSM8K, MATH, StrategyQA, ARC-Challenge, FOLIO
- **5 Models**: DS-R1-Qwen-7B, DS-R1-Llama-8B, DS-R1-Qwen-14B, QwQ-32B, DS-R1-Qwen-32B
- **3 Novel Metrics**: SIG, CNS, RFI
- **6-Category Failure Taxonomy**
- **5 Perturbation Tests**: Early answering, mistake injection, step shuffling, step deletion, paraphrasing
- **5 Ablation Studies**: Temperature, CoT length, perturbation type, prompt format, model scaling
- **4+ Result Tables** and **8+ Visualizations**